# 01 Data Collection

## Objective
The goal of this notebook is to:
- test the connection to the API-Football service,
- identify the Czech top division in the API,
- download data for the available historical seasons,
- save raw API responses season by season,
- create a basic match-level dataset for each season,
- combine all seasons into one raw dataset for downstream preprocessing.

## Rolling backtest setup
Based on API availability, the project uses the following rolling backtest design:

- Train: 2022 → Test: 2023
- Train: 2022–2023 → Test: 2024

## Expected outputs

### Raw season folders
`data/raw/czech_liga/{season}/`

For each season, the notebook saves:
- `leagues.json`
- `teams.json`
- `fixtures.json`
- `standings.json`
- `matches_basic.csv`

### Combined outputs
`data/interim/czech_liga/`

- `matches_all_seasons_raw.csv`
- `season_summary.csv`

**Project setup**

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_DIR)
print("Source directory exists:", SRC_DIR.exists())

Project root: C:\Users\cerve\Desktop\DP\football_prediction
Source directory: C:\Users\cerve\Desktop\DP\football_prediction\src
Source directory exists: True


**Imports**

In [2]:
import pandas as pd

from config import SETTINGS, RAW_DIR, INTERIM_DIR, validate_settings
from api_client import APIFootballClient, resolve_league_id
from data_io import save_json, save_csv, ensure_dir
from preprocessing import (
    fixtures_json_to_df,
    keep_finished_matches,
    create_target_columns,
    basic_match_cleanup,
)

**Display settings**

In [3]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 100)

## 1. Configuration overview

We first inspect the key configuration values used in this notebook.

In [4]:
validate_settings()

print("API key loaded:", bool(SETTINGS.api_key))
print("API key preview:", SETTINGS.api_key[:6] + "..." if SETTINGS.api_key else None)
print("Base URL:", SETTINGS.base_url)
print("Country filter:", SETTINGS.league_country)
print("League name hint:", SETTINGS.league_name_hint)
print("Seasons to download:", SETTINGS.seasons)
print("Rolling test seasons:", SETTINGS.rolling_test_seasons)

API key loaded: True
API key preview: 11fa5e...
Base URL: https://v3.football.api-sports.io
Country filter: Czech-Republic
League name hint: Czech Liga
Seasons to download: (2022, 2023, 2024)
Rolling test seasons: (2023, 2024)


## 2. API connection test

We create the API client and verify that the connection works correctly.

In [5]:
client = APIFootballClient(
    api_key=SETTINGS.api_key,
    base_url=SETTINGS.base_url,
    request_sleep_seconds=SETTINGS.request_sleep_seconds,
    max_retries=SETTINGS.max_retries,
    retry_backoff_seconds=SETTINGS.retry_backoff_seconds,
)

test_response = client.test_connection()
print("Connection successful.")
print("Number of countries returned:", len(test_response.get("response", [])))

Connection successful.
Number of countries returned: 171


## 3. Resolve the league ID

We identify the Czech top division using one reference season.

**Use the latest season as reference**

In [6]:
reference_season = max(SETTINGS.seasons)
print("Reference season:", reference_season)

reference_leagues_json = client.get_leagues(
    country=SETTINGS.league_country,
    season=reference_season,
)

print("League records returned:", len(reference_leagues_json.get("response", [])))

Reference season: 2024
League records returned: 15


**Preview league candidates**

In [7]:
league_rows = []

for item in reference_leagues_json.get("response", []):
    league_rows.append({
        "league_id": item.get("league", {}).get("id"),
        "league_name": item.get("league", {}).get("name"),
        "country": item.get("country", {}).get("name"),
        "type": item.get("league", {}).get("type"),
    })

league_df = pd.DataFrame(league_rows)
league_df

,league_id,league_name,country,type
0,345,Czech Liga,Czech-Republic,League
1,346,FNL,Czech-Republic,League
2,669,1. Liga Women,Czech-Republic,League
3,349,3. liga - MSFL,Czech-Republic,League
4,348,3. liga - CFL A,Czech-Republic,League
5,685,3. liga - CFL B,Czech-Republic,League
6,668,1. Liga U19,Czech-Republic,League
7,686,4. liga - Divizie F,Czech-Republic,League
8,354,4. liga - Divizie E,Czech-Republic,League
9,353,4. liga - Divizie D,Czech-Republic,League


**Resolve target league ID**

In [8]:
league_id = resolve_league_id(reference_leagues_json, SETTINGS.league_name_hint)
print("Resolved league ID:", league_id)

Resolved league ID: 345


## 4. Prepare output folders

We create the raw and interim output directories used in the notebook.

**Create output folders**

In [9]:
league_raw_root = RAW_DIR / "czech_liga"
league_interim_root = INTERIM_DIR / "czech_liga"

ensure_dir(league_raw_root)
ensure_dir(league_interim_root)

print("Raw root:", league_raw_root)
print("Interim root:", league_interim_root)

Raw root: C:\Users\cerve\Desktop\DP\football_prediction\data\raw\czech_liga
Interim root: C:\Users\cerve\Desktop\DP\football_prediction\data\interim\czech_liga


## 5. Download all seasons

For each season, we:
- download leagues, teams, fixtures, and standings,
- save raw JSON responses,
- convert fixtures to a basic match-level dataset,
- save `matches_basic.csv`.

**Download loop**

In [10]:
all_matches = []
season_summaries = []

preview_cols = [
    "fixture_id",
    "date",
    "season",
    "round",
    "home_team_name",
    "away_team_name",
    "home_goals",
    "away_goals",
    "result_1x2",
]

for season in SETTINGS.seasons:
    print("=" * 80)
    print(f"Processing season: {season}")

    season_raw_dir = league_raw_root / str(season)
    ensure_dir(season_raw_dir)

    teams_json = client.get_teams(
        league_id=league_id,
        season=season,
    )
    fixtures_json = client.get_fixtures(
        league_id=league_id,
        season=season,
    )
    standings_json = client.get_standings(
        league_id=league_id,
        season=season,
    )

    save_json(reference_leagues_json, season_raw_dir / "leagues.json")
    save_json(teams_json, season_raw_dir / "teams.json")
    save_json(fixtures_json, season_raw_dir / "fixtures.json")
    save_json(standings_json, season_raw_dir / "standings.json")

    fixtures_response_count = len(fixtures_json.get("response", []))
    print(f"Fixtures returned: {fixtures_response_count}")

    matches_df = fixtures_json_to_df(fixtures_json)

    if matches_df.empty:
        print(f"Warning: no fixture data available for season {season}. Skipping this season.")

        season_summary = {
            "season": season,
            "n_matches": 0,
            "n_home_teams": 0,
            "n_away_teams": 0,
            "date_min": None,
            "date_max": None,
            "n_home_wins": 0,
            "n_draws": 0,
            "n_away_wins": 0,
        }
        season_summaries.append(season_summary)

        save_csv(matches_df, season_raw_dir / "matches_basic.csv", index=False)
        continue

    matches_df = keep_finished_matches(matches_df)
    matches_df = basic_match_cleanup(matches_df)
    matches_df = create_target_columns(matches_df)

    save_csv(matches_df, season_raw_dir / "matches_basic.csv", index=False)

    all_matches.append(matches_df)

    outcome_counts = matches_df["result_1x2"].value_counts().to_dict() if len(matches_df) > 0 else {}

    season_summary = {
        "season": season,
        "n_matches": len(matches_df),
        "n_home_teams": matches_df["home_team_name"].nunique() if len(matches_df) > 0 else 0,
        "n_away_teams": matches_df["away_team_name"].nunique() if len(matches_df) > 0 else 0,
        "date_min": matches_df["date"].min() if len(matches_df) > 0 else None,
        "date_max": matches_df["date"].max() if len(matches_df) > 0 else None,
        "n_home_wins": outcome_counts.get("H", 0),
        "n_draws": outcome_counts.get("D", 0),
        "n_away_wins": outcome_counts.get("A", 0),
    }
    season_summaries.append(season_summary)

    print("Matches shape:", matches_df.shape)
    display(matches_df[preview_cols].head(5))

Processing season: 2022
Fixtures returned: 280
Matches shape: (280, 20)


,fixture_id,date,season,round,home_team_name,away_team_name,home_goals,away_goals,result_1x2
0,880111,2022-07-30 14:00:00,2022,Regular Season - 1,Zlin,Mlada Boleslav,0,0,D
1,880114,2022-07-30 14:00:00,2022,Regular Season - 1,Baník Ostrava,Sigma Olomouc,0,3,A
2,880112,2022-07-30 14:00:00,2022,Regular Season - 1,Zbrojovka Brno,Slovácko,2,2,D
3,880107,2022-07-30 17:00:00,2022,Regular Season - 1,Teplice,Plzen,2,2,D
4,880113,2022-07-31 14:00:00,2022,Regular Season - 1,FK Jablonec,Bohemians 1905,0,3,A


Processing season: 2023
Fixtures returned: 281
Matches shape: (281, 20)


,fixture_id,date,season,round,home_team_name,away_team_name,home_goals,away_goals,result_1x2
0,1049494,2023-07-22 13:00:00,2023,Regular Season - 1,Karviná,Zlin,4,1,H
1,1049495,2023-07-22 13:00:00,2023,Regular Season - 1,Teplice,Plzen,1,0,H
2,1049493,2023-07-22 13:00:00,2023,Regular Season - 1,Pardubice,Bohemians 1905,0,1,A
3,1049496,2023-07-22 16:00:00,2023,Regular Season - 1,Slavia Praha,Hradec Králové,2,0,H
4,1049497,2023-07-23 13:00:00,2023,Regular Season - 1,Mlada Boleslav,FK Jablonec,3,1,H


Processing season: 2024
Fixtures returned: 280
Matches shape: (280, 20)


,fixture_id,date,season,round,home_team_name,away_team_name,home_goals,away_goals,result_1x2
0,1210384,2024-07-19 17:00:00,2024,Regular Season - 1,Sparta Praha,Pardubice,2,1,H
1,1210383,2024-07-20 12:30:00,2024,Regular Season - 1,FK Jablonec,Mlada Boleslav,2,0,H
2,1210386,2024-07-20 15:00:00,2024,Regular Season - 1,Bohemians 1905,Baník Ostrava,2,1,H
3,1210380,2024-07-20 15:00:00,2024,Regular Season - 1,České Budějovice,Sigma Olomouc,0,2,A
4,1210385,2024-07-20 18:00:00,2024,Regular Season - 1,Dukla Praha,Plzen,1,3,A


## 6. Season-level summary

We inspect how many matches were collected in each season and verify the date range.

In [11]:
season_summary_df = pd.DataFrame(season_summaries)
season_summary_df

,season,n_matches,n_home_teams,n_away_teams,date_min,date_max,n_home_wins,n_draws,n_away_wins
0,2022,280,18,18,2022-07-30 14:00:00,2023-06-04 15:00:00,120,70,90
1,2023,281,18,18,2023-07-22 13:00:00,2024-06-02 15:30:00,127,71,83
2,2024,280,18,18,2024-07-19 17:00:00,2025-06-01 13:00:00,131,61,88


## 7. Combine all seasons into one dataset

We concatenate all season-level datasets into a single match-level file.

In [12]:
matches_all_df = pd.concat(all_matches, ignore_index=True)
matches_all_df["date"] = pd.to_datetime(matches_all_df["date"], errors="coerce")
matches_all_df = matches_all_df.sort_values(["date", "fixture_id"]).reset_index(drop=True)

print("Combined dataset shape:", matches_all_df.shape)

Combined dataset shape: (841, 20)


**Preview combined dataset**

In [13]:
matches_all_df.head(10)

,fixture_id,date,timestamp,status_long,status_short,league_id,league_name,season,round,home_team_id,home_team_name,away_team_id,away_team_name,home_goals,away_goals,halftime_home_goals,halftime_away_goals,result_1x2,goal_diff,total_goals
0,880111,2022-07-30 14:00:00,1659189600,Match Finished,FT,345,Czech Liga,2022,Regular Season - 1,603,Zlin,640,Mlada Boleslav,0,0,0,0,D,0,0
1,880112,2022-07-30 14:00:00,1659189600,Match Finished,FT,345,Czech Liga,2022,Regular Season - 1,3733,Zbrojovka Brno,3719,Slovácko,2,2,2,1,D,0,4
2,880114,2022-07-30 14:00:00,1659189600,Match Finished,FT,345,Czech Liga,2022,Regular Season - 1,3713,Baník Ostrava,2250,Sigma Olomouc,0,3,0,1,A,-3,3
3,880107,2022-07-30 17:00:00,1659200400,Match Finished,FT,345,Czech Liga,2022,Regular Season - 1,3720,Teplice,567,Plzen,2,2,1,0,D,0,4
4,880108,2022-07-31 14:00:00,1659276000,Match Finished,FT,345,Czech Liga,2022,Regular Season - 1,3724,Pardubice,3736,České Budějovice,0,2,0,1,A,-2,2
5,880109,2022-07-31 14:00:00,1659276000,Match Finished,FT,345,Czech Liga,2022,Regular Season - 1,3723,Hradec Králové,560,Slavia Praha,1,0,0,0,H,1,1
6,880113,2022-07-31 14:00:00,1659276000,Match Finished,FT,345,Czech Liga,2022,Regular Season - 1,1122,FK Jablonec,3714,Bohemians 1905,0,3,0,1,A,-3,3
7,880110,2022-07-31 17:00:00,1659286800,Match Finished,FT,345,Czech Liga,2022,Regular Season - 1,628,Sparta Praha,782,Slovan Liberec,1,2,0,2,A,-1,3
8,880117,2022-08-06 14:00:00,1659794400,Match Finished,FT,345,Czech Liga,2022,Regular Season - 2,782,Slovan Liberec,3720,Teplice,5,1,3,1,H,4,6
9,880118,2022-08-06 14:00:00,1659794400,Match Finished,FT,345,Czech Liga,2022,Regular Season - 2,567,Plzen,3724,Pardubice,2,1,1,0,H,1,3


**Tail preview**

In [14]:
matches_all_df.tail(10)

,fixture_id,date,timestamp,status_long,status_short,league_id,league_name,season,round,home_team_id,home_team_name,away_team_id,away_team_name,home_goals,away_goals,halftime_home_goals,halftime_away_goals,result_1x2,goal_diff,total_goals
831,1372304,2025-05-24 14:00:00,1748095200,Match Finished,FT,345,Czech Liga,2024,Championship Round - 5,567,Plzen,1122,FK Jablonec,4,1,3,0,H,3,5
832,1372305,2025-05-24 14:00:00,1748095200,Match Finished,FT,345,Czech Liga,2024,Championship Round - 5,628,Sparta Praha,2250,Sigma Olomouc,1,1,0,0,D,0,2
833,1375253,2025-05-25 11:30:00,1748172600,Match Finished,FT,345,Czech Liga,2024,Middle Play-offs - Final,3723,Hradec Králové,3714,Bohemians 1905,2,0,1,0,H,2,2
834,1372318,2025-05-25 14:00:00,1748181600,Match Finished,FT,345,Czech Liga,2024,Relegation Round - 5,640,Mlada Boleslav,3719,Slovácko,2,2,0,0,D,0,4
835,1372319,2025-05-25 14:00:00,1748181600,Match Finished,FT,345,Czech Liga,2024,Relegation Round - 5,3720,Teplice,3724,Pardubice,3,0,1,0,H,3,3
836,1372320,2025-05-25 14:00:00,1748181600,Match Finished,FT,345,Czech Liga,2024,Relegation Round - 5,3715,Dukla Praha,3736,České Budějovice,2,1,2,1,H,1,3
837,1376688,2025-05-28 16:00:00,1748448000,Match Finished,FT,345,Czech Liga,2024,Relegation Round,3724,Pardubice,3722,Chrudim,2,0,0,0,H,2,2
838,1376690,2025-05-28 16:00:00,1748448000,Match Finished,FT,345,Czech Liga,2024,Relegation Round,7379,Vyškov,3715,Dukla Praha,0,0,0,0,D,0,0
839,1376689,2025-06-01 13:00:00,1748782800,Match Finished,FT,345,Czech Liga,2024,Relegation Round,3722,Chrudim,3724,Pardubice,1,0,0,0,H,1,1
840,1376691,2025-06-01 13:00:00,1748782800,Match Finished,PEN,345,Czech Liga,2024,Relegation Round,3715,Dukla Praha,7379,Vyškov,1,1,0,1,D,0,2


**Matches per season**

In [15]:
matches_per_season = (
    matches_all_df.groupby("season")
    .size()
    .rename("n_matches")
    .reset_index()
)

matches_per_season

,season,n_matches
0,2022,280
1,2023,281
2,2024,280


**Outcome distribution by season**

In [16]:
outcome_by_season = (
    matches_all_df.groupby(["season", "result_1x2"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

outcome_by_season

result_1x2,season,A,D,H
0,2022,90,70,120
1,2023,83,71,127
2,2024,88,61,131


## 8. Save combined outputs

We save:
- the full raw multi-season match dataset,
- the season-level summary table.

In [17]:
combined_matches_path = league_interim_root / "matches_all_seasons_raw.csv"
season_summary_path = league_interim_root / "season_summary.csv"
league_candidates_path = league_interim_root / "league_candidates_reference.csv"

save_csv(matches_all_df, combined_matches_path, index=False)
save_csv(season_summary_df, season_summary_path, index=False)
save_csv(league_df, league_candidates_path, index=False)

print("Saved:", combined_matches_path)
print("Saved:", season_summary_path)
print("Saved:", league_candidates_path)

Saved: C:\Users\cerve\Desktop\DP\football_prediction\data\interim\czech_liga\matches_all_seasons_raw.csv
Saved: C:\Users\cerve\Desktop\DP\football_prediction\data\interim\czech_liga\season_summary.csv
Saved: C:\Users\cerve\Desktop\DP\football_prediction\data\interim\czech_liga\league_candidates_reference.csv


## 9. Final summary

We conclude the notebook with a concise project-level summary.

In [18]:
summary = {
    "league_id": league_id,
    "country": SETTINGS.league_country,
    "league_name_hint": SETTINGS.league_name_hint,
    "seasons_downloaded": SETTINGS.seasons,
    "rolling_test_seasons": SETTINGS.rolling_test_seasons,
    "n_total_matches": len(matches_all_df),
    "date_min": str(matches_all_df["date"].min()),
    "date_max": str(matches_all_df["date"].max()),
    "output_raw_root": str(league_raw_root),
    "output_interim_root": str(league_interim_root),
}

summary

{'league_id': 345,
 'country': 'Czech-Republic',
 'league_name_hint': 'Czech Liga',
 'seasons_downloaded': (2022, 2023, 2024),
 'rolling_test_seasons': (2023, 2024),
 'n_total_matches': 841,
 'date_min': '2022-07-30 14:00:00',
 'date_max': '2025-06-01 13:00:00',
 'output_raw_root': 'C:\\Users\\cerve\\Desktop\\DP\\football_prediction\\data\\raw\\czech_liga',
 'output_interim_root': 'C:\\Users\\cerve\\Desktop\\DP\\football_prediction\\data\\interim\\czech_liga'}

## Notebook summary

This notebook successfully:
- connected to the API-Football service,
- identified the Czech top division,
- downloaded data for multiple seasons,
- saved season-level raw files,
- created a combined raw match dataset.

## Next step
The next notebook (`02_preprocessing_eda.ipynb`) will:
- load `matches_all_seasons_raw.csv`,
- inspect data quality across all seasons,
- check duplicates and missing values,
- verify the class distribution,
- prepare a clean interim dataset for feature engineering.